## ABOUT

In this notebook, I implemented a simplified **Transformer Encoder Block** from scratch using **NumPy** to understand its internal working without relying on deep learning frameworks.

The implementation includes:
- Tokenization and random word embeddings
- Sinusoidal positional encoding
- Scaled dot-product self-attention
- Residual connections
- Position-wise Feed Forward Network (FFN)
- (Optional) Layer Normalization

Through this implementation, I gained a deeper understanding of how contextual token representations are generated, how attention captures relationships between words, and how residual connections and FFNs contribute to stable and expressive feature learning.

> **Note:** This is an educational implementation and differs from the original Transformer encoder by simplifying or omitting components such as multi-head attention, dropout, and trainable optimization. The primary goal is to build intuition behind the encoder architecture rather than train a production-ready model.

In [ ]:
import numpy as np

class EncoderBlock:

  def __init__(self, text, embedding_dim=512, seed=42) -> None:
      np.random.seed(seed)    # to keep randomness parmanent

      self.text = text
      self.y_embedding = None
      self.tokens = text.lower().split()

      self.vocab = sorted(set(self.tokens))
      self.vocab = {word: idx for idx, word in enumerate(self.vocab)}

      # Random embedding matrix
      self.embedding_matrix = np.random.randn(len(self.vocab), embedding_dim)

      # Convert sentence to embeddings
      self.embeddings = np.array([
          self.embedding_matrix[self.vocab[word]]
          for word in self.tokens
      ])
      self.dmodel = self.embeddings.shape[1]

      # initialize weights for query, key and value matrix
      self.w_Q = np.random.randn(self.dmodel, self.dmodel)
      self.w_K = np.random.randn(self.dmodel, self.dmodel)
      self.w_V = np.random.randn(self.dmodel, self.dmodel)

      self.encode_position()

      # feed forward network
      self.d_ff = 4 * self.dmodel

      self.w1 = np.random.randn(self.dmodel, self.d_ff)
      self.b1 = np.zeros((1, self.d_ff))

      self.w2 = np.random.randn(self.d_ff, self.dmodel)
      self.b2 = np.zeros((1, self.dmodel))

      # Layer Normalization parameters
      self.gamma = np.ones((1, self.dmodel))
      self.beta = np.zeros((1, self.dmodel))
      self.eps = 1e-5

  # Positional encoding on Embeddings
  def encode_position(self):
    final_encoding = []
    positions = np.arange(len(self.tokens))

    for i in range(self.dmodel // 2):
      omega = 1 / (10000 ** ((2 * i) / self.dmodel))

      # getting encodings on sin / cos functions
      pos_encoding0 = np.sin(omega * positions)
      pos_encoding1 = np.cos(omega * positions)

      # appending encodings
      final_encoding.extend([pos_encoding0, pos_encoding1])

    encoding = np.array(final_encoding)

    # add positional encoding with word's embedding
    self.PE_vector = self.embeddings + encoding.T

  # Activation Function
  def relu(self, x):
    return np.maximum(0, x)

  # function to calculate Softmax
  def softmax(self, x):
      exp = np.exp(x - np.max(x, axis=-1, keepdims=True))
      return exp / np.sum(exp, axis=-1, keepdims=True)

  # Layers Normalization
  def layer_norm(self, x):
    # Mean of each token
    mean = np.mean(x, axis=-1, keepdims=True)

    # Variance of each token
    variance = np.var(x, axis=-1, keepdims=True)

    # Normalize
    x_hat = (x - mean) / np.sqrt(variance + self.eps)

    # Scale and shift
    return self.gamma * x_hat + self.beta

  def feed_forward(self, x):

    hidden = x @ self.w1 + self.b1
    hidden = self.relu(hidden)

    output = hidden @ self.w2 + self.b2
    return output

  # Main Block
  def encoderBlock(self):
    # Generate Q, K, V
    Q = self.PE_vector @ self.w_Q
    K = self.PE_vector @ self.w_K
    V = self.PE_vector @ self.w_V

    # Compute Attention Scores
    d_k = K.shape[1]
    scores = (Q @ K.T) / np.sqrt(d_k)

    # Softmax
    weights = self.softmax(scores)

    # Weighted Sum of Values + Normalization
    attention_output = self.PE_vector + weights @ V
    attention_output = self.layer_norm(attention_output)

    # Feed Forward Network
    ffn_output = self.feed_forward(attention_output)

    # Second Residual
    output = attention_output + ffn_output
    output = self.layer_norm(output)

    return output

In [ ]:
encoder = EncoderBlock("How are you")

In [ ]:
encoder.PE_vector.shape

(3, 512)

In [ ]:
y_embedding = encoder.encoderBlock()

In [ ]:
y_embedding

array([[ 1.88287842,  0.47578697,  0.57127749, ..., -1.69237451,
         0.34522701, -2.43578469],
       [ 0.87428848,  1.26699137,  1.71574553, ..., -0.30032403,
        -1.18812246, -1.86233293],
       [ 0.94841563,  1.25529041,  1.69558655, ..., -0.24184288,
        -1.20175281, -1.84647527]])

In [ ]:
y_embedding.shape

(3, 512)

In [ ]:
# Debugging
positions = np.arange(len(encoder.tokens))
final_encoding = []

for i in range(512 // 2):
  omega = 1 / (10000 ** ((2 * i) / 512))

  # getting encodings on sin / cos functions
  pos_encoding0 = np.sin(omega * positions)
  pos_encoding1 = np.cos(omega * positions)

  # appending encodings
  final_encoding.extend([pos_encoding0, pos_encoding1])

np.array(final_encoding).shape

(512, 3)

In [ ]:
encoder.w_Q.shape

(512, 512)